## Solución Taller 02 - Optimización de Redes Metabólicas

In [ ]:
# Librerías a utilizar en el taller
import numpy as np
from itertools import combinations
from gurobipy import Model, GRB

### Pregunta a)

El objetivo de la pregunta es resolver el problema explorando todas las soluciones posibles y evaluando su valor en la función objetivo.

El primer paso es transformar el sistema en su forma canónica a:

<!-- $$
\begin{array}{l}
\text{s.t.} \\
\end{array}
\begin{array}{c}
\max_{x_1, x_2, x_3} \quad 3x_1 + x_2 + 31x_3 \\[1em]
6x_1 + 9x_2 + 26x_3 \leq 26 \\
3x_1 + 2x_2 - 26x_3 \leq 20 \\
x_1, x_2, x_3 \geq 0
\end{array}
$$ -->


$$
\begin{array}{l}
\text{s.t.} \\
\end{array}
\begin{array}{c}
\max_{x_1, x_2, x_3, x_4, x_5} \ \ 3x_1 + x_2 + 31x_3  \\[1em]
6x_1 + 9x_2 + 26x_3 + x_4 = 26 \\
3x_1 + 2x_2 - 26x_3 + x_5 = 20 \\
x_1, x_2, x_3, x_4, x_5 \geq 0
\end{array}
$$

Luego, dado que son 2 restricciones, las soluciones factibles tendran 2 variables básicas, por lo que hay un total de $\binom{5}{2} = 10$.

La condición para que sea factible, es que sean positivas para pertenecer al dominio.


In [16]:
# Matriz de restricciones
A = np.array([[6, 9, 26, 1, 0],
              [3, 2, -26, 0, 1]])

b = np.array([26, 20])

# Enumerar combinaciones de variables básicas
cols = [0, 1, 2, 3, 4]
BFS_posibles = list(combinations(cols, 2))

BFS = []

for sol_idx in BFS_posibles:
    B  = A[:, sol_idx]         # Extraigo solo las columnas de las variables básicas
    xB = np.linalg.solve(B, b) # Resuelvo el sistema de ecuaciones para las variables básicas
    
    if np.all(xB >= 0):        # Si la solución es positiva, entonces es una BFS
        x_full  = np.zeros(5)
        x_full[list(sol_idx)] = xB
        BFS.append(x_full)

BFS = np.array(BFS)

print(f"Número de BFS: {BFS.shape[0]}")
print(BFS)

# Evaluar función objetivo
c = np.array([3, 1, 31, 0, 0])
valores_obj = BFS @ c

ix_opt  = np.argmax(valores_obj)
max_opt = valores_obj[ix_opt]

print(f"\nValor óptimo: {max_opt}")
print(f"Solución óptima encontrada: {BFS[ix_opt]}")


Número de BFS: 4
[[ 4.33333333  0.          0.          0.          7.        ]
 [ 0.          2.88888889  0.          0.         14.22222222]
 [ 0.          0.          1.          0.         46.        ]
 [ 0.          0.          0.         26.         20.        ]]

Valor óptimo: 31.0
Solución óptima encontrada: [ 0.  0.  1.  0. 46.]


### Pregunta b)

Se resuelve ahora el problema usando Gurobi, para comprobar que se obtiene la misma solución que en a).

In [29]:
model = Model()
x1 = model.addVar(lb=0, ub=1e6, name="x1") # Se pone un limite superior grande para evitar problemas al indicar infinito positivo
x2 = model.addVar(lb=0, ub=1e6, name="x2")
x3 = model.addVar(lb=0, ub=1e6, name="x3")
x4 = model.addVar(lb=0, ub=1e6, name="x4")
x5 = model.addVar(lb=0, ub=1e6, name="x5")

# Definición de restricciones
model.addConstr(6*x1 + 9*x2 + 26*x3 + x4 == 26, name="c1")
model.addConstr(3*x1 + 2*x2 - 26*x3 + x5 == 20, name="c2")

# Función objetivo (maximizar)
model.setObjective(3*x1 + x2 + 31*x3 + 0*x4 + 0*x5, GRB.MAXIMIZE)

# sin output (opcional)
model.setParam('OutputFlag', 0)

# Resolver
model.optimize()

# Solución óptima
xopt = [x1.X, x2.X, x3.X, x4.X, x5.X]
fopt = model.ObjVal

print("Solución óptima:", xopt)
print("Valor óptimo:", fopt)

# Precios sombra (valores duales)
for constr in model.getConstrs():
    print(f"Precio sombra de {constr.ConstrName}: {constr.Pi:.3f}")


Solución óptima: [0.0, 0.0, 1.0, 0.0, 46.0]
Valor óptimo: 31.0
Precio sombra de c1: 1.192
Precio sombra de c2: -0.000


### Pregunta c)

Ahora se deben interpretar los precios sombra. Por definición, un precio sombra indica cuánto aumentaría/disminuiría la función objetivo si se aumenta en 1 unidad el lado derecho de una restricción activa.

In [34]:
# Obtener precios sombra
precios_sombra  = [constr.Pi for constr in model.getConstrs()]
max_prsombra    = max(precios_sombra)
ix_max_prsombra = precios_sombra.index(max_prsombra)

print(f"El aumento esperado en la función objetivo es de: {max_prsombra:.3f}, al aumentar en 1 la restricción {ix_max_prsombra+1}")
print(f"El nuevo valor de la función objetivo debería ser:{fopt + max_prsombra : .3f}")

El aumento esperado en la función objetivo es de: 1.192, al aumentar en 1 la restricción 1
El nuevo valor de la función objetivo debería ser: 32.192


In [ ]:
# Comprobación resolviendo el problema con lado derecho modificado

model2 = Model()
x1 = model2.addVar(lb=0, ub=1e6, name="x1")
x2 = model2.addVar(lb=0, ub=1e6, name="x2")
x3 = model2.addVar(lb=0, ub=1e6, name="x3")
x4 = model2.addVar(lb=0, ub=1e6, name="x4")
x5 = model2.addVar(lb=0, ub=1e6, name="x5")

# Definición de restricciones
model2.addConstr(6*x1 + 9*x2 + 26*x3 + x4 == 26 + 1, name="c1") # Se aumenta en 1 el lado derecho
model2.addConstr(3*x1 + 2*x2 - 26*x3 + x5 == 20, name="c2")

# Función objetivo (maximizar)
model2.setObjective(3*x1 + x2 + 31*x3 + 0*x4 + 0*x5, GRB.MAXIMIZE)

# sin output (opcional)
model2.setParam('OutputFlag', 0)

# Resolver
model2.optimize()

print(f"Nuevo valor de la función objet ivo: {model2.ObjVal:.3f}")


Nuevo valor de la función objetivo: 32.192


### Pregunta d)

Por último, se solicita formular el problema dual y resolverlo. 

El problema en su forma dual es el siguiente:

$$
\begin{array}{l}
\text{s.t.} \\
\end{array}
\begin{array}{c}
\min_{y_1, y_2} \ \ 26y_1 + 20y_2  \\[1em]
6y_1 + 3y_2 \geq 3 \\
9y_1 + 2y_2 \geq 1 \\
26y_1 - 26y_2 \geq 31 \\
y_1, y_2 \geq 0
\end{array}
$$


In [49]:
# Problema dual
dual = Model()

# Añadir variables del dual
y1 = dual.addVar(lb=0, ub=1e6, name="y1")
y2 = dual.addVar(lb=0, ub=1e6, name="y2")

# Definir función objetivo del dual
dual.setObjective(26*y1 + 20*y2, GRB.MINIMIZE)

# Añadir restricciones del dual
dual.addConstr(6*y1 + 3*y2   >= 3,  name="d1")
dual.addConstr(9*y1 + 2*y2   >= 1,  name="d2")
dual.addConstr(26*y1 - 26*y2 >= 31, name="d3")

# sin output (opcional)
dual.setParam('OutputFlag', 0)

# Resolver
dual.optimize()

xopt_dual = [y1.X, y2.X]
fopt_dual = dual.ObjVal

print("Solución óptima (dual):", [round(val, 3) for val in xopt_dual])
print("Valor óptimo del dual:", fopt_dual)
print("Precios sombra del dual:", [constr.Pi for constr in dual.getConstrs()])


Solución óptima (dual): [1.192, 0.0]
Valor óptimo del dual: 31.0
Precios sombra del dual: [0.0, 0.0, 1.0]


Se puede observar que la solución del dual es idéntica a los precios sombra del problema primal, y viceversa :D